# Final comparison

1. Validation accuracy of every model (the table for the presentation)
2. Do the models make the same mistakes?
3. Kaggle files for every combination of models, sorted by expected score

Run the 4 model notebooks first. Each one saves its predictions in `<model>/outputs/`.

## 1. Setup

In [1]:
import os
import sys

# The notebook is in a sub-folder. Go to the project folder (where common.py is).
if not os.path.exists("common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import itertools

import numpy as np
import pandas as pd

import common

MODELS = {                       # name -> folder
    "k-NN": "knn",
    "Decision Tree": "decision_tree",
    "Random Forest": "random_forest",
    "Neural Network": "neural_network",
}
TASKS = ["image", "sentiment"]
os.makedirs("final/submissions", exist_ok=True)

## 2. Load the validation predictions

Every model used the same validation rows (`common.stratified_split`), so we can compare them row by row.

In [2]:
y_val = {}
for task in TASKS:
    _, y, _, _ = common.load_task(task)
    _, val_idx = common.stratified_split(y)
    y_val[task] = y[val_idx]

val_pred = {task: {} for task in TASKS}
for task in TASKS:
    for name, folder in MODELS.items():
        val_path = f"{folder}/outputs/{task}_val_pred.npy"
        test_path = f"{folder}/outputs/{task}_test_pred.csv"
        if os.path.exists(val_path) and os.path.exists(test_path):
            val_pred[task][name] = np.load(val_path)
        else:
            print(f"missing: {name} ({task}) -> run {folder}/{folder}.ipynb to the end first")

## 3. Validation accuracy of every model

In [3]:
rows = []
for name in MODELS:
    row = {"Method": name}
    for task in TASKS:
        if name in val_pred[task]:
            row[f"{task} accuracy"] = round(common.accuracy(y_val[task], val_pred[task][name]), 4)
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv("final/validation_table.csv", index=False)
table

,Method,image accuracy,sentiment accuracy
0,k-NN,0.6008,0.6000
1,Decision Tree,0.1678,0.6692
2,Random Forest,0.3685,0.6708
3,Neural Network,0.4567,0.6867


## 4. Do the models make the same mistakes?

For every pair of models: how many validation examples **both** get wrong, and how many only one of them gets wrong.

In [4]:
for task in TASKS:
    wrong = {name: pred != y_val[task] for name, pred in val_pred[task].items()}
    print(f"\n{task} ({len(y_val[task])} validation examples)")
    for a, b in itertools.combinations(wrong, 2):
        both = (wrong[a] & wrong[b]).sum()
        only_a = (wrong[a] & ~wrong[b]).sum()
        only_b = (~wrong[a] & wrong[b]).sum()
        print(f"   {a} vs {b}: both wrong {both}, only {a} wrong {only_a}, only {b} wrong {only_b}")
    if wrong:
        all_wrong = np.logical_and.reduce(list(wrong.values())).sum()
        print(f"   wrong for every model: {all_wrong}")


image (4000 validation examples)
   k-NN vs Decision Tree: both wrong 1472, only k-NN wrong 125, only Decision Tree wrong 1857
   k-NN vs Random Forest: both wrong 1379, only k-NN wrong 218, only Random Forest wrong 1147
   k-NN vs Neural Network: both wrong 1270, only k-NN wrong 327, only Neural Network wrong 903
   Decision Tree vs Random Forest: both wrong 2350, only Decision Tree wrong 979, only Random Forest wrong 176
   Decision Tree vs Neural Network: both wrong 2025, only Decision Tree wrong 1304, only Neural Network wrong 148
   Random Forest vs Neural Network: both wrong 1819, only Random Forest wrong 707, only Neural Network wrong 354
   wrong for every model: 1104

sentiment (2400 validation examples)
   k-NN vs Decision Tree: both wrong 582, only k-NN wrong 378, only Decision Tree wrong 212
   k-NN vs Random Forest: both wrong 571, only k-NN wrong 389, only Random Forest wrong 219
   k-NN vs Neural Network: both wrong 580, only k-NN wrong 380, only Neural Network wrong 17

## 5. Kaggle files

We write one file for **every combination** (image model + sentiment model), with its expected score:

`expected score = (image validation accuracy + sentiment validation accuracy) / 2`

This is the same formula Kaggle uses. The files are sorted from best to worst.

* Pick the file to submit using the **validation** score, not the Kaggle score (the Kaggle public score uses only half of the test data).
* On Kaggle you can select **2 submissions** for the final ranking. A good choice: the best file, and the best file that uses only one model.

In [5]:
val_acc = {task: {name: common.accuracy(y_val[task], pred) for name, pred in val_pred[task].items()}
           for task in TASKS}

rows = []
for image_model in val_acc["image"]:
    for sent_model in val_acc["sentiment"]:
        expected = (val_acc["image"][image_model] + val_acc["sentiment"][sent_model]) / 2
        rows.append({"image model": image_model, "sentiment model": sent_model,
                     "expected score": round(expected, 4)})
combos = pd.DataFrame(rows).sort_values("expected score", ascending=False).reset_index(drop=True)


def short(name):
    return name.lower().replace(" ", "_").replace("-", "")       # "k-NN" -> "knn"


files = []
for _, row in combos.iterrows():
    image = pd.read_csv(f"{MODELS[row['image model']]}/outputs/image_test_pred.csv")
    sentiment = pd.read_csv(f"{MODELS[row['sentiment model']]}/outputs/sentiment_test_pred.csv")
    path = f"final/submissions/image_{short(row['image model'])}__sentiment_{short(row['sentiment model'])}.csv"
    common.make_submission(image["id"], image["label"], sentiment["id"], sentiment["label"], path)
    files.append(path)
combos["file"] = files
combos.to_csv("final/submissions/expected_scores.csv", index=False)
combos

,image model,sentiment model,expected score,file
0,k-NN,Neural Network,0.6437,final/submissions/image_knn__sentiment_neural_...
1,k-NN,Random Forest,0.6358,final/submissions/image_knn__sentiment_random_...
2,k-NN,Decision Tree,0.6350,final/submissions/image_knn__sentiment_decisio...
3,k-NN,k-NN,0.6004,final/submissions/image_knn__sentiment_knn.csv
4,Neural Network,Neural Network,0.5717,final/submissions/image_neural_network__sentim...
5,Neural Network,Random Forest,0.5638,final/submissions/image_neural_network__sentim...
6,Neural Network,Decision Tree,0.5630,final/submissions/image_neural_network__sentim...
7,Neural Network,k-NN,0.5284,final/submissions/image_neural_network__sentim...
8,Random Forest,Neural Network,0.5276,final/submissions/image_random_forest__sentime...
9,Random Forest,Random Forest,0.5197,final/submissions/image_random_forest__sentime...
